# News Headline Classification Using Machine Learning

Classify news headlines into **World, Sports, Business, Sci/Tech** with TF-IDF + classical ML (Naive Bayes, Logistic Regression, Linear SVM) on the AG News dataset.

Run order: **Run All**. No API keys needed — the dataset downloads from public CSVs inside the notebook.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

MIRROR = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv"
train_full = pd.read_csv(f"{MIRROR}/train.csv", header=None, names=['label', 'title', 'description'])
test = pd.read_csv(f"{MIRROR}/test.csv", header=None, names=['label', 'title', 'description'])
train = pd.concat([d.sample(3000, random_state=42) for _, d in train_full.groupby('label')])
print(train.shape, test.shape)

In [ ]:
LABEL_MAP = {1: 'World', 2: 'Sports', 3: 'Business', 4: 'Sci/Tech'}
for df in (train, test):
    df['text'] = df['title'].fillna('') + ' ' + df['description'].fillna('')
    df['label_name'] = df['label'].map(LABEL_MAP)

train['label_name'].value_counts().plot(kind='bar', title='Headlines per category (train)')
plt.ylabel('count')
plt.show()
print(train['label_name'].value_counts())

In [ ]:
target_names = ['World', 'Sports', 'Business', 'Sci/Tech']
models = {
    'naive_bayes': MultinomialNB(),
    'logistic_regression': LogisticRegression(max_iter=1000, C=4.0),
    'linear_svm': LinearSVC(C=1.0),
}

best_name, best_acc, best_pipe = None, 0.0, None
for name, clf in models.items():
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=20000, ngram_range=(1, 2),
                                    stop_words='english', sublinear_tf=True)),
        ('clf', clf),
    ])
    pipe.fit(train['text'], train['label'])
    acc = accuracy_score(test['label'], pipe.predict(test['text']))
    print(f'{name:20s} test accuracy = {acc:.4f}')
    if acc > best_acc:
        best_name, best_acc, best_pipe = name, acc, pipe
print(f'\nBest model: {best_name} ({best_acc:.4f})')

In [ ]:
pred = best_pipe.predict(test['text'])
print(classification_report(test['label'], pred, target_names=target_names))
ConfusionMatrixDisplay.from_predictions(test['label'], pred, display_labels=target_names, xticks_rotation=45)
plt.title(f'Confusion matrix — {best_name}')
plt.show()

In [ ]:
demo = [
    'NASA unveils new Mars rover mission',
    'Sensex surges 500 points on strong earnings',
    'India wins T20 World Cup in thrilling final',
    'UN summit reaches breakthrough climate deal',
]
for h in demo:
    print(f'{h}  ->  {LABEL_MAP[best_pipe.predict([h])[0]]}')